In [ ]:
from nautilus_trader.accounting.accounts.margin import MarginAccount
help(MarginAccount)


In [ ]:
import os
import pandas as pd
from datetime import datetime

# Nautilus imports for constructing data and instruments
from nautilus_trader.model.data import QuoteTick
from nautilus_trader.model.identifiers import InstrumentId
from nautilus_trader.model.objects import Price, Quantity, Currency
from nautilus_trader.core.datetime import dt_to_unix_nanos
from nautilus_trader.persistence.catalog.parquet import ParquetDataCatalog
from nautilus_trader.model.instruments.option_contract import OptionContract
from nautilus_trader.model.identifiers import InstrumentId, Symbol, Venue
from nautilus_trader.model.enums import AssetClass, OptionKind
from nautilus_trader.core.datetime import dt_to_unix_nanos
from datetime import datetime, timezone

import logging

# Enable debug-level logs
logging.basicConfig(level=logging.DEBUG)

# --- Set directories
csv_dir = "data/nse"
catalog_dir = "catalog"
catalog_meta_dir = "catalog-meta"
os.makedirs(catalog_dir, exist_ok=True)
os.makedirs(catalog_meta_dir, exist_ok=True)

# --- Initialize Parquet catalog to store core data (quote ticks and instruments)
catalog = ParquetDataCatalog(catalog_dir)

# --- Storage lists
all_ticks = []                 # Stores QuoteTick instances
instrument_ids = set()         # Keeps track of all unique instrument IDs
meta_records = []              # Stores auxiliary/meta data as dict rows

# --- Loop through each CSV file in the input directory
for fname in os.listdir(csv_dir):
    if not fname.endswith(".csv") or fname.startswith("._"):
        continue

    print(f"Reading {fname}")
    df = pd.read_csv(os.path.join(csv_dir, fname))

    # Convert timestamps to nanoseconds since epoch (UNIX time)
    df["timestamp"] = pd.to_datetime(df["timestamp"], format="%m/%d/%y %H:%M")
    df["timestamp"] = df["timestamp"].apply(dt_to_unix_nanos)

    # --- Parse rows into QuoteTick objects
    for _, row in df.iterrows():
        try:
             
            iid_str = row["symbol"].strip()
            parts = iid_str.split(".")
            symbol = parts[0]               # e.g. BANKNIFTY
            venue = parts[1]                # e.g. NSE
            type = parts[2]                 # e.g. OPTION
            expiry_raw = parts[3]           # e.g. 26Jun2025
            strike = float(parts[4])
            right = parts[5].upper()        # CALL or PUT
            symbolplus = f"{symbol}.{type}.{expiry_raw}.{int(strike)}.{right}"
            instrument_id = InstrumentId(symbol=Symbol(symbolplus), venue=Venue(venue))

            # Setup price and size values
            bid_val = float(row["bid"])
            ask_val = float(row["ask"])
            bid_precision = 2
            ask_precision = 2
            bid_size_val = float(row.get("bid_size", 1))
            ask_size_val = float(row.get("ask_size", 1))
            size_precision = 0

            # Create QuoteTick object
            tick = QuoteTick(
                instrument_id=instrument_id,
                ts_event=int(row["timestamp"]),
                ts_init=int(row["timestamp"]),
                bid_price=Price(int(bid_val * 10**bid_precision), bid_precision),
                ask_price=Price(int(ask_val * 10**ask_precision), ask_precision),
                bid_size=Quantity(int(bid_size_val * 10**size_precision), size_precision),
                ask_size=Quantity(int(ask_size_val * 10**size_precision), size_precision),
            )

            all_ticks.append(tick)
          
            instrument_ids.add(instrument_id)

            # Save the meta fields separately
            meta_records.append({
                "instrument_id": f"{symbolplus}.{venue}",
                "timestamp": row["timestamp"],
                "impliedVolatility": row.get("impliedVolatility"),
                "openInterest": row.get("openInterest"),
                "last": row.get("last"),
                "pChange": row.get("pChange"),
            })

        except Exception as e:
            print(f"⚠️ Skipping row due to tick parsing error: {e}")

# --- Build dummy instruments using OptionContract.from_dict
# This will store instrument metadata needed by the engine
print(list(instrument_ids)[0])
print(list(meta_records)[0])
print(all_ticks[0])
# Example loop for just 1 instrument ID
dummy_instruments = []
for iid in instrument_ids:
    try:
        print(f"🔍 Processing: {iid.value}")
        parts = iid.value.split(".")
        symbol = parts[0]               # e.g. BANKNIFTY
        type = parts[1] 
        expiry_raw = parts[2]           # e.g. 26Jun2025
        strike = float(parts[3])
        right = parts[4].upper()        # CALL or PUT
        venue = parts[5]               # e.g. NSE
        symbolplus = f"{symbol}.{type}.{expiry_raw}.{int(strike)}.{right}"

        expiry_dt = datetime.strptime(expiry_raw, "%d%b%Y")
        expiry_utc = expiry_dt.replace(hour=15, minute=30, tzinfo=timezone.utc)
        now_utc = datetime.now(timezone.utc)

        option = OptionContract(
            instrument_id=InstrumentId(symbol=Symbol(symbolplus ), venue=Venue(venue)),
            raw_symbol=Symbol(symbol),
            asset_class=AssetClass.INDEX,     # Same as example
            exchange=venue,
            currency=Currency.from_str("INR"),
            price_precision=2,
            price_increment=Price.from_str("0.05"),
            multiplier=Quantity.from_int(15),
            lot_size=Quantity.from_int(1),
            underlying=f"{symbol}.{venue}.INDEX",
            option_kind=OptionKind[right],
            strike_price=Price.from_str(str(strike)),
            activation_ns=0,  # Set to 0 to make it always active
            expiration_ns=dt_to_unix_nanos(expiry_utc),
            ts_event=dt_to_unix_nanos(now_utc),
            ts_init=dt_to_unix_nanos(now_utc),
        )

        print(f"✅ Created OptionContract: {option.id}")
        dummy_instruments.append(option)
    except Exception as e:
        print(f"⚠️ Skipping {iid.value} due to instrument creation error: {e}")

catalog.write_data(dummy_instruments)
catalog.write_data(all_ticks)
# --- Write meta-data (IV, OI, etc.) to separate catalog-meta folder
meta_df = pd.DataFrame(meta_records)
meta_df.to_parquet(os.path.join(catalog_meta_dir, "tick_metadata.parquet"), index=False)

print(f"\n✅ Written {len(all_ticks)} quote ticks to {catalog_dir}/")
print(f"✅ Written {len(meta_df)} metadata records to {catalog_meta_dir}/")
print(f"✅ Written {len(dummy_instruments)} dummy instruments")

# Create a DataFrame of instrument metadata
instrument_meta = [{
    "instrument_id": str(inst.id),
    "symbol": str(inst.raw_symbol),
    "strike": float(inst.strike_price.value / (10 ** inst.strike_price.precision)),
    "expiry": inst.expiration_ns,
    "option_kind": inst.option_kind.name,
    "venue": inst.exchange,
    "activation_ns": inst.activation_ns,
} for inst in dummy_instruments]

df_instruments = pd.DataFrame(instrument_meta)
df_instruments.to_parquet(os.path.join(catalog_meta_dir, "instruments.parquet"), index=False)

print(f"✅ Written {len(dummy_instruments)} dummy instruments to instrument metadata")

In [ ]:

# Step 1: Add strategy to engine
from decimal import Decimal
from nautilus_trader.backtest.engine import BacktestEngine
from nautilus_trader.config import BacktestEngineConfig, LoggingConfig
from nautilus_trader.model import InstrumentId, TraderId
from nautilus_trader.model.enums import AccountType, OmsType
from nautilus_trader.model.objects import Money
from nautilus_trader.model.currencies import INR
from nautilus_trader.model.identifiers import Venue
from nautilus_trader.persistence.catalog.parquet import ParquetDataCatalog

from src.strategies.my_nse_strategy import MyNSEStrategy, MyNSEStrategyConfig

# Step 1: Backtest engine config with debug logging
engine_config = BacktestEngineConfig(
    trader_id=TraderId("BACKTEST_TRADER-001"),
    logging=LoggingConfig(log_level="DEBUG"),
)

engine = BacktestEngine(config=engine_config)


In [ ]:
# Step 1.5: Load instruments and quote ticks from catalog

import subprocess
import sys

# Get the path to the python executable from the current environment
python_executable = sys.executable

# Define the path to your script
script_path = "0 clean.py"

# Run the script as a separate process
result = subprocess.run(
    [python_executable, script_path],
    capture_output=True,  # Capture stdout and stderr
    text=True,            # Decode output as text
    check=False           # Don't raise an exception on non-zero exit codes
)

# Print the output and any errors
print("--- STDOUT ---")
print(result.stdout)
print("--- STDERR ---")
print(result.stderr)

if result.returncode == 0:
    print(f"\nScript '{script_path}' executed successfully.")
else:
    print(f"\nScript '{script_path}' finished with exit code: {result.returncode}")


In [ ]:
# Step 2: Load instruments and quote ticks from catalog
data_catalog = ParquetDataCatalog("catalog")  # folder with .parquet files

# Load all instruments from the catalog
original_instruments = data_catalog.instruments()
print(f"Loaded {len(original_instruments)} original instruments from catalog.")

# **THE FIX**: Iterate through instruments and correct the activation timestamp
corrected_instruments = []
for inst in original_instruments:
    if isinstance(inst, OptionContract):
        # Re-create the OptionContract with activation_ns set to 0
        inst_dict = OptionContract.to_dict(inst)
        inst_dict["activation_ns"] = 0
        corrected_inst = OptionContract.from_dict(inst_dict)
        corrected_instruments.append(corrected_inst)
    else:
        # If not an option contract, just append as-is
        corrected_instruments.append(inst)
print(f"Corrected {len(corrected_instruments)} instruments with activation_ns=0.")

# Use the corrected instruments
instruments = corrected_instruments

# Load all quote ticks from the catalog
ticks = data_catalog.quote_ticks()
print(f"Loaded {len(ticks)} quote ticks from catalog.")

In [ ]:
#step 2.5
# Debug: Check what instruments are in the data
print("=== INSTRUMENT ANALYSIS ===")
instrument_counts = {}
for tick in ticks:
    inst_id = str(tick.instrument_id)
    if inst_id not in instrument_counts:
        instrument_counts[inst_id] = 0
    instrument_counts[inst_id] += 1

print("Instruments in data:")
for inst_id, count in instrument_counts.items():
    print(f"  - {inst_id}: {count} ticks")

# Check if our target instrument has data
target_inst = "BANKNIFTY.OPT.26Jun2025.64000.CALL.NSE"
if target_inst in instrument_counts:
    print(f"✅ Target instrument {target_inst} has {instrument_counts[target_inst]} ticks")
else:
    print(f"❌ Target instrument {target_inst} has NO ticks!")
    print("Available instruments:")
    for inst_id in instrument_counts.keys():
        if "64000" in inst_id:
            print(f"  - Found similar: {inst_id}")

In [ ]:
# Step 3: Add venue, instrument, and data to engine
venue_id = "NSE"
existing_venues = [str(v) for v in engine.list_venues()]
print ("Why")
if venue_id not in existing_venues:
    engine.add_venue(
        venue=Venue(venue_id),
        oms_type=OmsType.NETTING,
        account_type=AccountType.MARGIN,
        starting_balances=[Money(1_000_000, INR)],
        base_currency=INR,
        default_leverage=Decimal(1),
    )
# Add all instruments
for inst in instruments:
    try:
        engine.add_instrument(inst)
    except Exception as e:
        print(f"⚠️ Error adding instrument {inst.instrument_id}: {e}")
print("✅ Added instruments to engine.")

# Add all quote ticks (remove the try-except to see any errors)
engine.add_data(ticks)
print(f"✅ Added {len(ticks)} quote ticks to the engine.")

In [ ]:
# Fix instrument activation timestamps
corrected_instruments = []
for inst in instruments:
    if hasattr(inst, 'activation_ns'):
        inst_dict = OptionContract.to_dict(inst)
        inst_dict['activation_ns'] = 0
        corrected_inst = OptionContract.from_dict(inst_dict)
        corrected_instruments.append(corrected_inst)
    else:
        corrected_instruments.append(inst)

# Replace the original instruments with corrected ones
instruments = corrected_instruments

In [ ]:
# Step 3.5
# Debug: Check if target instrument and ticks exist
target_instrument_id = InstrumentId.from_str("BANKNIFTY.OPT.26Jun2025.64000.CALL.NSE")

# Check instruments
target_instrument = None
for inst in instruments:
    if inst.id == target_instrument_id:  # Changed from inst.instrument_id to inst.id
        target_instrument = inst
        break
print(f"Target instrument found: {target_instrument is not None}")

# Check ticks
target_ticks = [t for t in ticks if t.instrument_id == target_instrument_id]
print(f"Ticks for target instrument: {len(target_ticks)}")

if len(target_ticks) > 0:
    print(f"First tick: {target_ticks[0]}")
    print(f"Last tick: {target_ticks[-1]}")
else:
    print("❌ NO TICKS FOUND FOR TARGET INSTRUMENT!")

In [ ]:
# Step 4: Add your strategy
import sys
sys.path.append('src')
import yaml
from strategies.my_nse_strategy import MyNSEStrategy, MyNSEStrategyConfig
from nautilus_trader.model import InstrumentId

# Load YAML
with open("src/config/nse_strategy_config.yaml") as f:
    yaml_config = yaml.safe_load(f)

# Create config with just the essential parameters
config = MyNSEStrategyConfig(
    instrument_id=InstrumentId.from_str("BANKNIFTY.OPT.26Jun2025.64000.CALL.NSE"),
    # All other parameters will use their default values from the class
)
strategy = MyNSEStrategy(config)

# Debug: Check strategy before adding
print(f"Strategy ID: {strategy.id}")
print(f"Strategy state: {strategy.state}")
print(f"Strategy config: {strategy.config}")

# Add strategy to engine
engine.add_strategy(strategy)

# Debug: Check if strategy was added
print(f"Strategy added to engine")
print(f"Engine trader ID: {engine.trader_id}")

In [ ]:
# Step 4.5
# Debug: Check strategy setup
print("=== STRATEGY DEBUGGING ===")
print(f"Strategy ID: {strategy.id}")
print(f"Strategy instrument: {strategy.config.instrument_id}")

# Check if strategy is properly configured
print(f"Strategy has on_start method: {hasattr(strategy, 'on_start')}")
print(f"Strategy has subscribe_quote_ticks method: {hasattr(strategy, 'subscribe_quote_ticks')}")

# Check engine state (using correct methods)
print(f"Engine trader ID: {engine.trader_id}")

# Check if data was added successfully
print(f"Total instruments we loaded: {len(instruments)}")
print(f"Total ticks we loaded: {len(ticks)}")

# Check strategy configuration
print(f"Strategy config instrument: {strategy.config.instrument_id}")
print(f"Target instrument ID: {target_instrument_id}")
print(f"Strategy and target match: {strategy.config.instrument_id == target_instrument_id}")

In [ ]:
# Debug: Check engine and strategy state before running
print("=== PRE-RUN DEBUGGING ===")
print(f"Engine trader ID: {engine.trader_id}")
print(f"Strategy state: {strategy.state}")
print(f"Strategy ID: {strategy.id}")

# Check if strategy is properly registered
print(f"Strategy is registered: {hasattr(strategy, 'trader_id')}")
if hasattr(strategy, 'trader_id'):
    print(f"Strategy trader ID: {strategy.trader_id}")

# Check if strategy has the required components
print(f"Strategy has cache: {hasattr(strategy, 'cache')}")
print(f"Strategy has portfolio: {hasattr(strategy, 'portfolio')}")
print(f"Strategy has order_factory: {hasattr(strategy, 'order_factory')}")

In [ ]:
# Add debugging before running
print("=== PRE-RUN DEBUGGING ===")
print(f"Engine trader ID: {engine.trader_id}")
print(f"Strategy state: {strategy.state}")
print(f"Strategy ID: {strategy.id}")
print(f"Strategy trader ID: {strategy.trader_id}")
print(f"Strategy has cache: {strategy.cache is not None}")
print(f"Strategy has portfolio: {strategy.portfolio is not None}")
print(f"Strategy has order_factory: {strategy.order_factory is not None}")

# Check if strategy is subscribed to data
print(f"Strategy subscribed instruments: {list(strategy.subscribed_instruments()) if hasattr(strategy, 'subscribed_instruments') else 'No subscribed_instruments method'}")

# Try to explicitly start the strategy
print("\n=== EXPLICITLY STARTING STRATEGY ===")
try:
    strategy.start()
    print("Strategy started successfully")
except Exception as e:
    print(f"Error starting strategy: {e}")

print(f"Strategy state after start: {strategy.state}")

# Run the backtest
print("\n=== RUNNING BACKTEST ===")
engine.run()

print("\n=== POST-RUN DEBUGGING ===")
print(f"Engine trader ID: {engine.trader_id}")
print(f"Strategy state: {strategy.state}")
print(f"Strategy ID: {strategy.id}")
print(f"Strategy trader ID: {strategy.trader_id}")
print(f"Strategy has cache: {strategy.cache is not None}")
print(f"Strategy has portfolio: {strategy.portfolio is not None}")
print(f"Strategy has order_factory: {strategy.order_factory is not None}")

# Check if strategy received any data
print(f"\nStrategy received quote ticks: {hasattr(strategy, '_quote_tick_count') and strategy._quote_tick_count > 0}")
if hasattr(strategy, '_quote_tick_count'):
    print(f"Quote tick count: {strategy._quote_tick_count}")

# Check engine results
print(f"\nEngine events count: {len(engine.event_log)}")
print(f"Engine orders count: {len(engine.order_log)}")
print(f"Engine positions count: {len(engine.position_log)}")

# Check if strategy has any orders or positions
print(f"\nStrategy orders: {len(strategy.cache.orders()) if strategy.cache else 0}")
print(f"Strategy positions: {len(strategy.cache.positions()) if strategy.cache else 0}")

In [ ]:
# Add debugging to check if strategy receives quote ticks
print("=== CHECKING STRATEGY QUOTE TICK RECEPTION ===")

# Check if strategy has any quote tick count
if hasattr(strategy, '_quote_tick_count'):
    print(f"Strategy quote tick count: {strategy._quote_tick_count}")
else:
    print("Strategy has no _quote_tick_count attribute")

# Check strategy cache for any data
if strategy.cache:
    print(f"Strategy cache orders: {len(strategy.cache.orders())}")
    print(f"Strategy cache positions: {len(strategy.cache.positions())}")
    print(f"Strategy cache has data: {strategy.cache is not None}")
else:
    print("Strategy has no cache")

# Check if strategy has any trades
print(f"Strategy trades: {len(strategy.trades) if hasattr(strategy, 'trades') else 'No trades attribute'}")

# Check engine results using correct attributes
print(f"\nEngine cache orders: {len(engine.cache.orders())}")
print(f"Engine cache positions: {len(engine.cache.positions())}")
print(f"Engine data count: {len(engine.data)}")
print(f"Engine iteration: {engine.iteration}")

In [ ]:
# Get the actual trading results
from nautilus_trader.model.identifiers import Venue

print("\n=== TRADING RESULTS ===")

# Strategy trades
if hasattr(strategy, 'trades') and strategy.trades:
    print(f"Strategy trades count: {len(strategy.trades)}")
    print("Strategy trades:")
    for i, trade in enumerate(strategy.trades[:5]):
        print(f"  Trade {i+1}: {trade}")
else:
    print("No strategy trades found.")

# Engine orders
engine_orders = engine.cache.orders()
print(f"\nEngine orders count: {len(engine_orders)}")
if engine_orders:
    print("Engine orders (first 5):")
    for i, order in enumerate(engine_orders[:5]):
        print(f"  Order {i+1}: {order}")

# Engine positions
engine_positions = engine.cache.positions()
print(f"\nEngine positions count: {len(engine_positions)}")
if engine_positions:
    print("Engine positions:")
    for i, position in enumerate(engine_positions):
        print(f"  Position {i+1}: {position}")

# Portfolio details
venue_id = Venue("NSE")
account = engine.portfolio.account(venue_id)

if account:
    print(f"\nPortfolio Balance: {account.balance}")
    print(f"Portfolio Equity: {account.equity}")
    print(f"Margin Used: {account.margin_used}")
    print(f"Margin Available: {account.margin_available}")
else:
    print("\nNo account found for venue NSE.")


In [ ]:
# View trades
strategy.trades[:5]

In [ ]:
# 5. Analyze results
# You can now access engine.orders, engine.trades, engine.portfolio, etc.
print("Orders:", engine.orders)
print("Trades:", engine.trades)
print("Portfolio:", engine.portfolio)

In [ ]:
engine.reset()

In [ ]:
engine.reset()

In [ ]:
engine.dispose()